# 🏗️ Arquitetura Gold - Data Lakehouse

## 📊 Visão Geral

Documentação completa da **camada Gold** do Data Lakehouse, implementada com modelagem dimensional (Star Schema) seguindo boas práticas de Data Warehousing.

### Objetivo
A camada Gold fornece **dados prontos para consumo analítico** com modelagem dimensional otimizada para:
* Consultas de Business Intelligence
* Dashboards e relatórios
* Análises OLAP
* Machine Learning (features prontas)

### Padrões Implementados
* ✅ **Modelagem Star Schema** (Kimball)
* ✅ **Surrogate Keys** com xxhash64()
* ✅ **Dimensões Conformadas**
* ✅ **Bridge Tables** para relacionamentos N:N
* ✅ **SCD Type 1** (overwrite)
* ✅ **Materialized Views** (DLT)
* ✅ **Auto Optimize + Liquid Clustering**

---

## 💾 Catálogos Gold

### Estrutura de Catálogos

```
gold_dev/
├── dm_camara_deputados/
├── dm_carros/
├── dm_ibge/
├── dm_open_meteo/
└── dm_pokemon/

gold_prod/
├── dm_camara_deputados/
├── dm_carros/
├── dm_ibge/
├── dm_open_meteo/
└── dm_pokemon/
```

### Convenções de Nomenclatura

| Tipo | Prefixo | Exemplo |
|------|---------|--------|
| Dimensão | `dim_` | `dim_deputado`, `dim_partido` |
| Fato | `fact_` | `fact_despesas`, `fact_votacoes` |
| Bridge | `bridge_` | `bridge_pokemon_abilities` |
| Chave Substituta | `sk_` | `sk_deputado`, `sk_data` |
| Chave Natural | `id_` | `id_municipio`, `id_pokemon` |
| Schema Gold | `dm_` | `dm_camara_deputados`, `dm_carros` |
| Schema Silver | `ds_` | `ds_camara_deputados`, `ds_carros` |

---

## 🏛️ Schema 1: dm_camara_deputados

### 📁 Visão Geral
Dados políticos da Câmara dos Deputados do Brasil - despesas parlamentares e votações legislativas.

**Schema Gold:** `gold_{env}.dm_camara_deputados`  
**Schema Silver:** `silver_{env}.ds_camara_deputados`

### 📦 Dimensões (6)

#### 1. **dim_deputado**
Deputados federais brasileiros

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `sk_deputado` | BIGINT | Chave substituta | **PK** |
| `id` | BIGINT | ID natural da API | NK |
| `nome` | STRING | Nome completo | |
| `sigla_partido` | STRING | Sigla do partido | |
| `sigla_uf` | STRING | UF representada | |
| `id_legislatura` | BIGINT | ID da legislatura | |
| `url_foto` | STRING | URL da foto oficial | |
| `email` | STRING | E-mail de contato | |

**Cardinalidade:** ~600 deputados ativos

---

#### 2. **dim_partido**
Partidos políticos brasileiros

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `id` | BIGINT | ID do partido | **PK** |
| `sigla` | STRING | Sigla (PT, PSDB, etc.) | |
| `nome` | STRING | Nome completo | |
| `uri` | STRING | URI da API | |
| `status` | STRING | Status atual | |

**Cardinalidade:** ~30 partidos

---

#### 3. **dim_tempo**
Calendário temporal

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `sk_data` | INT | YYYYMMDD | **PK** |
| `data` | DATE | Data completa | |
| `ano` | INT | Ano | |
| `mes` | INT | Mês (1-12) | |
| `trimestre` | INT | Trimestre (1-4) | |
| `semestre` | INT | Semestre (1-2) | |
| `nome_mes` | STRING | Nome do mês | |
| `dia_semana` | INT | Dia da semana (1-7) | |
| `nome_dia_semana` | STRING | Nome do dia | |
| `eh_fim_semana` | BOOLEAN | Flag fim de semana | |

**Origem:** Derivada de datas de despesas e votações

---

#### 4. **dim_tipo_despesa**
Tipos de despesas parlamentares

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `sk_tipo_despesa` | BIGINT | Chave substituta | **PK** |
| `tipo_despesa` | STRING | Descrição da despesa | |
| `cod_tipo_documento` | BIGINT | Código do documento | |
| `tipo_documento` | STRING | Tipo de documento | |

**Exemplos:** Combustíveis, Passagens Aéreas, Telefonia, etc.

---

#### 5. **dim_fornecedor**
Fornecedores de produtos/serviços

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `sk_fornecedor` | BIGINT | Chave substituta | **PK** |
| `cnpj_cpf` | STRING | CNPJ ou CPF | NK |
| `nome_fornecedor` | STRING | Razão social | |

---

#### 6. **dim_orgao**
Órgãos legislativos

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `sk_orgao` | BIGINT | Chave substituta | **PK** |
| `id_orgao` | STRING | ID do órgão | NK |
| `sigla_orgao` | STRING | Sigla | |
| `uri_orgao` | STRING | URI da API | |

**Exemplos:** Plenário, CCJC, CAE, etc.

---

### 📊 Fatos (2)

#### 1. **fact_despesas**
Despesas da cota parlamentar

| Coluna | Tipo | Descrição | Tipo |
|--------|------|-----------|------|
| `sk_deputado` | BIGINT | FK -> dim_deputado | **FK** |
| `sk_data` | INT | FK -> dim_tempo | **FK** |
| `sk_tipo_despesa` | BIGINT | FK -> dim_tipo_despesa | **FK** |
| `sk_fornecedor` | BIGINT | FK -> dim_fornecedor | **FK** |
| `cod_documento` | STRING | Código do documento | **NK** |
| `num_documento` | STRING | Número do documento | |
| `valor_documento` | DECIMAL(15,2) | Valor bruto | **MÉTRICA** |
| `valor_liquido` | DECIMAL(15,2) | Valor líquido | **MÉTRICA** |
| `valor_glosa` | DECIMAL(15,2) | Valor glosado | **MÉTRICA** |
| `num_ressarcimento` | STRING | Número ressarcimento | |
| `url_documento` | STRING | URL do documento | |
| `cod_lote` | BIGINT | Código do lote | |
| `parcela` | BIGINT | Número da parcela | |

**Granularidade:** 1 linha = 1 documento fiscal
**Volume:** Milhões de registros

---

#### 2. **fact_votacoes**
Votações do Plenário e Comissões

| Coluna | Tipo | Descrição | Tipo |
|--------|------|-----------|------|
| `sk_votacao` | BIGINT | Chave substituta | **PK** |
| `sk_orgao` | BIGINT | FK -> dim_orgao | **FK** |
| `sk_data` | INT | FK -> dim_tempo | **FK** |
| `data_hora_registro` | TIMESTAMP | Data/hora completa | |
| `proposicao_cod_tipo` | STRING | Código tipo proposição | |
| `proposicao_numero` | STRING | Número proposição | |
| `proposicao_ano` | STRING | Ano proposição | |
| `uri_proposicao` | STRING | URI da proposição | |
| `aprovacao` | INT | 1=Aprovado, 0=Rejeitado | **MÉTRICA** |
| `objeto_votacao` | STRING | Descrição/ementa | |

**Granularidade:** 1 linha = 1 votação

---

### 🔗 Relacionamentos

```
dim_deputado (1) --< (N) fact_despesas
dim_tempo (1) --< (N) fact_despesas
dim_tipo_despesa (1) --< (N) fact_despesas
dim_fornecedor (1) --< (N) fact_despesas

dim_orgao (1) --< (N) fact_votacoes
dim_tempo (1) --< (N) fact_votacoes
```

---

## 🚗 Schema 2: dm_carros (FIPE)

### 📁 Visão Geral
Preços de veículos da tabela FIPE para análise de mercado automobilístico.

**Schema Gold:** `gold_{env}.dm_carros`  
**Schema Silver:** `silver_{env}.ds_carros`

### 📦 Dimensões (4)

#### 1. **dim_referencia**
Referências temporais FIPE

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `id_referencia` | STRING | ID da referência | **PK** |
| `ano_mes_referencia` | STRING | Formato ano_mes | |
| `mes_referencia` | INT | Mês | |
| `ano_referencia` | INT | Ano | |

---

#### 2. **dim_marca**
Marcas de veículos

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `id_marca` | STRING | ID da marca | **PK** |
| `nome_marca` | STRING | Nome (Fiat, VW, etc.) | |

---

#### 3. **dim_modelo**
Modelos por marca

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `id_marca` | STRING | FK -> dim_marca | **FK** |
| `id_modelo` | STRING | ID do modelo | **PK** |
| `nome_modelo` | STRING | Nome (Gol, Civic, etc.) | |

---

#### 4. **dim_veiculo**
Veículos completos (versões)

| Coluna | Tipo | Descrição | Chave |
|--------|------|-----------|-------|
| `sk_veiculo` | BIGINT | Chave substituta | **PK** |
| `id_marca` | STRING | FK -> dim_marca | **FK** |
| `nome_marca` | STRING | Nome da marca | |
| `id_modelo` | STRING | FK -> dim_modelo | **FK** |
| `nome_modelo` | STRING | Nome do modelo | |
| `modelo_fipe` | STRING | Descrição FIPE completa | |
| `ano_modelo` | INT | Ano do modelo | |
| `combustivel` | STRING | Tipo de combustível | |
| `codigo_fipe` | STRING | Código FIPE | NK |
| `id_ano` | STRING | ID do ano API | |

---

### 📊 Fatos (1)

#### **fact_preco_fipe**
Preços históricos FIPE

| Coluna | Tipo | Descrição | Tipo |
|--------|------|-----------|------|
| `sk_veiculo` | BIGINT | FK -> dim_veiculo | **FK** |
| `id_referencia` | STRING | FK -> dim_referencia | **FK** |
| `valor_fipe` | DECIMAL(12,2) | Preço em R$ | **MÉTRICA** |

**Granularidade:** 1 linha = 1 veículo em 1 referência

---

## 🌎 Schema 3: dm_ibge

### 📁 Visão Geral
Geografia territorial brasileira hierarquizada segundo IBGE.

**Schema Gold:** `gold_{env}.dm_ibge`  
**Schema Silver:** `silver_{env}.ds_ibge`

### 📦 Dimensões Hierarquizadas (5)

#### Hierarquia Geográfica

```
Região (5)
 └── Estado/UF (27)
     └── Município (5.570)
         └── Distrito
             └── Subdistrito
```

#### 1. **dim_regiao** (nível 1)

| Coluna | Tipo | Chave |
|--------|------|-------|
| `id_regiao` | BIGINT | **PK** |
| `nome_regiao` | STRING | |
| `sigla_regiao` | STRING | |

**Valores:** Norte, Nordeste, Centro-Oeste, Sudeste, Sul

---

#### 2. **dim_estado** (nível 2)

| Coluna | Tipo | Chave |
|--------|------|-------|
| `id_estado` | BIGINT | **PK** |
| `nome_estado` | STRING | |
| `sigla_estado` | STRING | |
| `id_regiao` | BIGINT | **FK** |
| `nome_regiao` | STRING | |
| `sigla_regiao` | STRING | |

**Cardinalidade:** 27 UFs

---

#### 3. **dim_municipio** (nível 3)

Tabela completa com **toda a hierarquia** desnormalizada:

| Coluna | Tipo | Nível |
|--------|------|--------|
| `id_municipio` | BIGINT | **PK** |
| `nome_municipio` | STRING | 3 |
| `id_microrregiao` | BIGINT | 3.1 |
| `nome_microrregiao` | STRING | 3.1 |
| `id_mesorregiao` | BIGINT | 3.2 |
| `nome_mesorregiao` | STRING | 3.2 |
| `id_regiao_imediata` | BIGINT | 3.3 |
| `nome_regiao_imediata` | STRING | 3.3 |
| `id_regiao_intermediaria` | BIGINT | 3.4 |
| `nome_regiao_intermediaria` | STRING | 3.4 |
| `id_estado` | BIGINT | 2 (FK) |
| `nome_estado` | STRING | 2 |
| `sigla_estado` | STRING | 2 |
| `id_regiao` | BIGINT | 1 (FK) |
| `nome_regiao` | STRING | 1 |
| `sigla_regiao` | STRING | 1 |

**Cardinalidade:** 5.570 municípios

---

#### 4. **dim_distrito** (nível 4)

| Coluna | Tipo | Chave |
|--------|------|-------|
| `id_distrito` | BIGINT | **PK** |
| `nome_distrito` | STRING | |
| `id_municipio` | BIGINT | **FK** |
| ... hierarquia acima ... | | |

---

#### 5. **dim_subdistrito** (nível 5)

| Coluna | Tipo | Chave |
|--------|------|-------|
| `id_subdistrito` | BIGINT | **PK** |
| `nome_subdistrito` | STRING | |
| `id_distrito` | BIGINT | **FK** |
| ... hierarquia acima ... | | |

---

### 🔗 Uso das Dimensões

Estas dimensões são **dimensões conformadas** que podem ser usadas por QUALQUER fato que tenha localização geográfica:

* fact_despesas (por UF do deputado)
* fact_clima (por capital)
* fact_vendas (por município)
* etc.

---

## ☁️ Schema 4: dm_open_meteo

### 📁 Visão Geral
Dados climáticos, previsões e qualidade do ar para capitais brasileiras.

**Schema Gold:** `gold_{env}.dm_open_meteo`  
**Schema Silver:** `silver_{env}.ds_open_meteo`

### 📦 Dimensões (2)

#### 1. **dim_localidade**
Capitais com coordenadas

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `sk_localidade` | BIGINT | **PK** |
| `id` | BIGINT | ID geocoding |
| `capital` | STRING | Nome da capital |
| `uf` | STRING | Sigla UF |
| `nome_oficial` | STRING | Nome oficial |
| `latitude` | DOUBLE | Latitude |
| `longitude` | DOUBLE | Longitude |
| `elevation` | DOUBLE | Elevação (m) |
| `timezone` | STRING | Fuso horário |
| `populacao` | BIGINT | População |
| `pais` | STRING | País |
| `codigo_pais` | STRING | Código ISO |

---

#### 2. **dim_tempo_clima**
Calendário com estações

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `sk_data` | INT | **PK** (YYYYMMDD) |
| `data` | DATE | Data |
| `ano` | INT | Ano |
| `mes` | INT | Mês |
| `trimestre` | INT | Trimestre |
| `semestre` | INT | Semestre |
| `nome_mes` | STRING | Nome do mês |
| `dia_ano` | INT | Dia do ano (1-366) |
| `semana_ano` | INT | Semana do ano |
| `dia_semana` | INT | Dia da semana |
| `nome_dia_semana` | STRING | Nome do dia |
| `eh_fim_semana` | BOOLEAN | Flag fim de semana |
| `estacao` | STRING | **Verão/Outono/Inverno/Primavera** |

---

### 📊 Fatos (3)

#### 1. **fact_clima_historico**
Dados climáticos históricos (horários)

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `sk_localidade` | BIGINT | **FK** |
| `sk_data` | INT | **FK** |
| `data_hora` | TIMESTAMP | Data/hora completa |
| `temperatura_2m` | DOUBLE | Temp em °C |
| `temperatura_aparente` | DOUBLE | Sensação térmica |
| `precipitacao` | DOUBLE | Precipitação (mm) |
| `chuva` | DOUBLE | Chuva (mm) |
| `velocidade_vento_10m` | DOUBLE | Vento (km/h) |
| `direcao_vento_10m` | DOUBLE | Direção vento |
| `umidade_relativa_2m` | DOUBLE | Umidade (%) |
| `pressao_nivel_mar` | DOUBLE | Pressão (hPa) |

---

#### 2. **fact_previsao_tempo**
Previsões diárias

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `sk_localidade` | BIGINT | **FK** |
| `sk_data` | INT | **FK** |
| `temperatura_max` | DOUBLE | Temp máx (°C) |
| `temperatura_min` | DOUBLE | Temp mín (°C) |
| `temperatura_media` | DOUBLE | Temp média (°C) |
| `precipitacao_total` | DOUBLE | Precipitação (mm) |
| `velocidade_vento_max` | DOUBLE | Vento máx (km/h) |
| `codigo_clima` | STRING | Código WMO |
| `nascer_sol` | TIMESTAMP | Nascer do sol |
| `por_sol` | TIMESTAMP | Pôr do sol |

---

#### 3. **fact_qualidade_ar**
Poluentes atmosféricos e índices AQI

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `sk_localidade` | BIGINT | **FK** |
| `sk_data` | INT | **FK** |
| `data_hora` | TIMESTAMP | Data/hora |
| `monoxido_carbono` | DOUBLE | CO (µg/m³) |
| `dioxido_nitrogenio` | DOUBLE | NO₂ (µg/m³) |
| `ozonio` | DOUBLE | O₃ (µg/m³) |
| `pm2_5` | DOUBLE | PM2.5 (µg/m³) |
| `pm10` | DOUBLE | PM10 (µg/m³) |
| `dioxido_enxofre` | DOUBLE | SO₂ (µg/m³) |
| `indice_europeu_aqi` | INT | AQI Europeu (1-5) |
| `indice_us_aqi` | INT | AQI EUA (0-500) |
| `indice_uv` | DOUBLE | Índice UV |

---

## 🐾 Schema 5: dm_pokemon

### 📁 Visão Geral
Dados do universo Pokémon com relacionamentos complexos N:N.

**Schema Gold:** `gold_{env}.dm_pokemon`  
**Schema Silver:** `silver_{env}.ds_pokemon`

### 📦 Dimensões (6)

#### 1. **dim_pokemon**

| Coluna | Tipo | Chave |
|--------|------|-------|
| `id_pokemon` | INT | **PK** |
| `nome_pokemon` | STRING | |

---

#### 2. **dim_item**

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_item` | INT | **PK** |
| `nome_item` | STRING | Nome |
| `custo` | INT | Preço |
| `sprite_url` | STRING | URL sprite |

---

#### 3. **dim_location**

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_location` | INT | **PK** |
| `nome_location` | STRING | Nome |
| `id_regiao` | INT | ID região |
| `nome_regiao` | STRING | Nome região |

---

#### 4. **dim_ability**

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_ability` | INT | **PK** |
| `nome_ability` | STRING | Nome |
| `eh_habilidade_oculta` | BOOLEAN | Hidden ability |

---

#### 5. **dim_nature**

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_nature` | INT | **PK** |
| `nome_nature` | STRING | Nome |
| `stat_aumentado` | STRING | Stat + |
| `stat_diminuido` | STRING | Stat - |

---

#### 6. **dim_growth_rate**

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_growth_rate` | INT | **PK** |
| `nome_growth_rate` | STRING | Nome |
| `formula` | STRING | Fórmula EXP |

---

### 🌉 Bridge Tables (2)

#### 1. **bridge_pokemon_abilities**
Relacionamento N:N Pokemon ↔ Abilities

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_pokemon` | INT | **FK** |
| `id_ability` | INT | **FK** |
| `eh_habilidade_oculta` | BOOLEAN | Hidden |
| `slot` | INT | Slot (1-3) |

**Granularidade:** 1 pokémon pode ter 1-3 habilidades

---

#### 2. **bridge_pokemon_locations**
Relacionamento N:N Pokemon ↔ Locations

| Coluna | Tipo | Descrição |
|--------|------|------------|
| `id_pokemon` | INT | **FK** |
| `id_location_area` | INT | FK |
| `nome_location_area` | STRING | Área |
| `nivel_minimo` | INT | Nível mín |
| `nivel_maximo` | INT | Nível máx |
| `metodo_encontro` | STRING | Walk/Surf/Fish |
| `taxa_encontro` | INT | % chance |

**Granularidade:** 1 pokémon pode aparecer em múltiplas áreas

---

## 📊 Exemplos de Consultas Analíticas

### Exemplo 1: Despesas por Partido e UF

```sql
SELECT
  p.sigla AS partido,
  d.sigla_uf AS uf,
  COUNT(DISTINCT d.sk_deputado) AS qtd_deputados,
  SUM(f.valor_liquido) AS total_despesas,
  AVG(f.valor_liquido) AS media_despesa
FROM gold_prod.dm_camara_deputados.fact_despesas f
JOIN gold_prod.dm_camara_deputados.dim_deputado d
  ON f.sk_deputado = d.sk_deputado
JOIN gold_prod.dm_camara_deputados.dim_partido p
  ON d.sigla_partido = p.sigla
JOIN gold_prod.dm_camara_deputados.dim_tempo t
  ON f.sk_data = t.sk_data
WHERE t.ano = 2024
GROUP BY p.sigla, d.sigla_uf
ORDER BY total_despesas DESC;
```

---

### Exemplo 2: Evolução de Preços FIPE

```sql
SELECT
  m.nome_marca,
  mo.nome_modelo,
  v.ano_modelo,
  v.combustivel,
  r.ano_mes_referencia,
  f.valor_fipe
FROM gold_prod.dm_carros.fact_preco_fipe f
JOIN gold_prod.dm_carros.dim_veiculo v
  ON f.sk_veiculo = v.sk_veiculo
JOIN gold_prod.dm_carros.dim_marca m
  ON v.id_marca = m.id_marca
JOIN gold_prod.dm_carros.dim_modelo mo
  ON v.id_modelo = mo.id_modelo
JOIN gold_prod.dm_carros.dim_referencia r
  ON f.id_referencia = r.id_referencia
WHERE m.nome_marca = 'Volkswagen'
  AND mo.nome_modelo = 'Gol'
  AND v.ano_modelo = 2020
ORDER BY r.ano_mes_referencia;
```

---

### Exemplo 3: Análise Climática por Estação

```sql
SELECT
  l.capital,
  l.uf,
  t.estacao,
  AVG(f.temperatura_2m) AS temp_media,
  MAX(f.temperatura_2m) AS temp_maxima,
  MIN(f.temperatura_2m) AS temp_minima,
  SUM(f.precipitacao) AS precipitacao_total
FROM gold_prod.dm_open_meteo.fact_clima_historico f
JOIN gold_prod.dm_open_meteo.dim_localidade l
  ON f.sk_localidade = l.sk_localidade
JOIN gold_prod.dm_open_meteo.dim_tempo_clima t
  ON f.sk_data = t.sk_data
WHERE t.ano = 2024
GROUP BY l.capital, l.uf, t.estacao
ORDER BY l.capital, t.estacao;
```

---

### Exemplo 4: Pokémons por Habilidade

```sql
SELECT
  a.nome_ability AS habilidade,
  COUNT(DISTINCT b.id_pokemon) AS qtd_pokemon,
  COLLECT_LIST(p.nome_pokemon) AS lista_pokemon
FROM gold_prod.dm_pokemon.bridge_pokemon_abilities b
JOIN gold_prod.dm_pokemon.dim_pokemon p
  ON b.id_pokemon = p.id_pokemon
JOIN gold_prod.dm_pokemon.dim_ability a
  ON b.id_ability = a.id_ability
WHERE b.eh_habilidade_oculta = false
GROUP BY a.nome_ability
HAVING qtd_pokemon >= 10
ORDER BY qtd_pokemon DESC;
```

---

## 📊 Exemplos de Consultas Analíticas

### Exemplo 1: Despesas por Partido e UF

```sql
SELECT
  p.sigla AS partido,
  d.sigla_uf AS uf,
  COUNT(DISTINCT d.sk_deputado) AS qtd_deputados,
  SUM(f.valor_liquido) AS total_despesas,
  AVG(f.valor_liquido) AS media_despesa
FROM gold_prod.ds_camara_deputados.fact_despesas f
JOIN gold_prod.ds_camara_deputados.dim_deputado d
  ON f.sk_deputado = d.sk_deputado
JOIN gold_prod.ds_camara_deputados.dim_partido p
  ON d.sigla_partido = p.sigla
JOIN gold_prod.ds_camara_deputados.dim_tempo t
  ON f.sk_data = t.sk_data
WHERE t.ano = 2024
GROUP BY p.sigla, d.sigla_uf
ORDER BY total_despesas DESC;
```

---

### Exemplo 2: Evolução de Preços FIPE

```sql
SELECT
  m.nome_marca,
  mo.nome_modelo,
  v.ano_modelo,
  v.combustivel,
  r.ano_mes_referencia,
  f.valor_fipe
FROM gold_prod.ds_carros.fact_preco_fipe f
JOIN gold_prod.ds_carros.dim_veiculo v
  ON f.sk_veiculo = v.sk_veiculo
JOIN gold_prod.ds_carros.dim_marca m
  ON v.id_marca = m.id_marca
JOIN gold_prod.ds_carros.dim_modelo mo
  ON v.id_modelo = mo.id_modelo
JOIN gold_prod.ds_carros.dim_referencia r
  ON f.id_referencia = r.id_referencia
WHERE m.nome_marca = 'Volkswagen'
  AND mo.nome_modelo = 'Gol'
  AND v.ano_modelo = 2020
ORDER BY r.ano_mes_referencia;
```

---

### Exemplo 3: Análise Climática por Estação

```sql
SELECT
  l.capital,
  l.uf,
  t.estacao,
  AVG(f.temperatura_2m) AS temp_media,
  MAX(f.temperatura_2m) AS temp_maxima,
  MIN(f.temperatura_2m) AS temp_minima,
  SUM(f.precipitacao) AS precipitacao_total
FROM gold_prod.ds_open_meteo.fact_clima_historico f
JOIN gold_prod.ds_open_meteo.dim_localidade l
  ON f.sk_localidade = l.sk_localidade
JOIN gold_prod.ds_open_meteo.dim_tempo_clima t
  ON f.sk_data = t.sk_data
WHERE t.ano = 2024
GROUP BY l.capital, l.uf, t.estacao
ORDER BY l.capital, t.estacao;
```

---

### Exemplo 4: Pokémons por Habilidade

```sql
SELECT
  a.nome_ability AS habilidade,
  COUNT(DISTINCT b.id_pokemon) AS qtd_pokemon,
  COLLECT_LIST(p.nome_pokemon) AS lista_pokemon
FROM gold_prod.ds_pokemon.bridge_pokemon_abilities b
JOIN gold_prod.ds_pokemon.dim_pokemon p
  ON b.id_pokemon = p.id_pokemon
JOIN gold_prod.ds_pokemon.dim_ability a
  ON b.id_ability = a.id_ability
WHERE b.eh_habilidade_oculta = false
GROUP BY a.nome_ability
HAVING qtd_pokemon >= 10
ORDER BY qtd_pokemon DESC;
```

---

## 📋 Sumário Completo

### Estatísticas da Camada Gold

| Schema | Dimensões | Fatos | Bridge | Total |
|--------|------------|-------|--------|-------|
| **ds_camara_deputados** | 6 | 2 | 0 | **8** |
| **ds_carros** | 4 | 1 | 0 | **5** |
| **ds_ibge** | 5 | 0 | 0 | **5** |
| **ds_open_meteo** | 2 | 3 | 0 | **5** |
| **ds_pokemon** | 6 | 0 | 2 | **8** |
| **TOTAL** | **23** | **6** | **2** | **31** |

---

### Notebooks Gold Criados

1. 🏛️ `dlt_camara_deputados_gold.sql`
2. 🚗 `dlt_carros_gold.sql` (já existente, validado)
3. 🌎 `dlt_ibge_gold.sql`
4. ☁️ `dlt_open_meteo_gold.sql`
5. 🐾 `dlt_pokemon_gold.sql`

---

### Próximos Passos

#### 🔴 Alta Prioridade
1. **Criar pipelines DLT** para cada schema Gold
2. **Configurar schedules** para refresh automático
3. **Validar** dados após primeira execução
4. **Documentar** grants e permissões

#### 🟡 Média Prioridade
5. **Criar dashboards** de monitoramento de qualidade
6. **Implementar testes** automatizados (Great Expectations)
7. **Otimizar** consultas lentas identificadas

#### 🟢 Baixa Prioridade
8. **Criar views agregadas** para dashboards específicos
9. **Implementar SCD Type 2** para dimensões que mudam lentamente
10. **Adicionar** column-level lineage

---

### 📞 Contatos e Manutenção

**Responsável:** Data Engineering Team
**Atualizado em:** 2026-05-20
**Versão:** 1.0

---

### 📚 Referências

* [Databricks DLT Documentation](https://docs.databricks.com/delta-live-tables/index.html)
* [Kimball Dimensional Modeling](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
* [Liquid Clustering](https://docs.databricks.com/delta/clustering.html)

---